<a href="https://colab.research.google.com/github/v35001/aoid-team2-ahiyp/blob/main/11_regression.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
# ==============================================================================
# 1. 패키지 설치 및 구글 드라이브 연결, 원자료 불러오기
# ==============================================================================

# 필요한 패키지 먼저 설치
!pip -q install pyreadstat

from google.colab import drive
drive.mount('/content/drive')

import os
import numpy as np
import pandas as pd
import statsmodels.api as sm
import statsmodels.formula.api as smf
import pyreadstat
from IPython.display import display

# 데이터가 들어 있는 폴더 (구글 드라이브 루트 폴더로 수정)
DATA_DIR = '/content/drive/MyDrive'

# 파일 위치 자동 검색
file_path = os.path.join(DATA_DIR, 'hn24_all.sas7bdat')

if not os.path.exists(file_path):
    for root, dirs, files in os.walk(DATA_DIR):
        if 'hn24_all.sas7bdat' in files:
            file_path = os.path.join(root, 'hn24_all.sas7bdat')
            break

if not os.path.exists(file_path):
    raise FileNotFoundError(
        'hn24_all.sas7bdat 파일을 구글 드라이브에서 찾을 수 없습니다.'
    )

cols = [
    'age', 'sex', 'HE_DM_HbA1c', 'HE_HP', 'HE_obe',
    'incm', 'edu', 'BS3_1', 'BD1_11',
    'wt_itvex', 'kstrata', 'psu'
]

df, meta = pyreadstat.read_sas7bdat(
    file_path,
    usecols=cols
)

df.columns = df.columns.str.lower()

print('원자료 불러오기 완료')
print('파일 위치:', file_path)
print('원자료 크기:', df.shape)


# ==============================================================================
# 2. 변수 정의 및 전처리
# ==============================================================================

# 당뇨병: 0=비유병, 1=유병
df['dm'] = np.where(
    df['he_dm_hba1c'].isin([1, 2]), 0,
    np.where(df['he_dm_hba1c'] == 3, 1, np.nan)
)

# 고혈압: 0=비유병, 1=유병
df['hp'] = np.where(
    df['he_hp'].isin([1, 2, 3]), 0,
    np.where(df['he_hp'] == 4, 1, np.nan)
)

# 성별: 0=여자, 1=남자
df['sex_cat'] = np.where(
    df['sex'] == 2, 0,
    np.where(df['sex'] == 1, 1, np.nan)
)

# 연령: 만 19세 이상
df['age_num'] = np.where(
    df['age'] >= 19, df['age'], np.nan
)

# 비만: 0=비만 아님, 1=비만
df['obe_cat'] = np.where(
    df['he_obe'].isin([1, 2, 3]), 0,
    np.where(df['he_obe'].isin([4, 5, 6]), 1, np.nan)
)

# 소득수준
df['incm_cat'] = df['incm'].where(
    df['incm'].isin([1, 2, 3, 4]), np.nan
)

# 교육수준
df['edu_cat'] = df['edu'].where(
    df['edu'].isin([1, 2, 3, 4]), np.nan
)

# 흡연: 0=비흡연, 1=현재 흡연
df['smk_cat'] = np.where(
    df['bs3_1'].isin([3, 8]), 0,
    np.where(df['bs3_1'].isin([1, 2]), 1, np.nan)
)

# 음주: 0=월 1회 미만/안마심, 1=월 1회 이상
df['drk_cat'] = np.where(
    df['bd1_11'].isin([1, 2, 8]), 0,
    np.where(df['bd1_11'].isin([3, 4, 5, 6]), 1, np.nan)
)


# ==============================================================================
# 3. 연구 대상자 선정 및 결측치 처리
# ==============================================================================

# 만 19세 이상이며 당뇨병과 고혈압 상태가 확인되는 대상자
sub_df = df[
    (df['age'] >= 19) &
    (df['dm'].notna()) &
    (df['hp'].notna())
].copy()

# 회귀분석에 필요한 변수
adjust_vars = [
    'age_num', 'sex_cat', 'incm_cat',
    'edu_cat', 'smk_cat', 'drk_cat', 'obe_cat'
]

analysis_vars = ['dm', 'hp'] + adjust_vars + [
    'wt_itvex', 'kstrata', 'psu'
]

# 두 모형에 동일한 대상자 사용
reg_df = sub_df[analysis_vars].dropna().copy()

# 복합표본 가중치가 양수인 대상자 확인
reg_df = reg_df[reg_df['wt_itvex'] > 0].copy()

print('--- 분석 대상자 확인 ---')
print('Table 1 분석 대상자:', len(sub_df))
print('회귀분석 최종 대상자:', len(reg_df))
print('보정변수 결측 등으로 제외된 인원:', len(sub_df) - len(reg_df))

print('\n당뇨병 유병 여부')
print(reg_df['dm'].value_counts().sort_index())

print('\n고혈압 유병 여부')
print(reg_df['hp'].value_counts().sort_index())


# ==============================================================================
# 4. 회귀분석 모형 설정
# ==============================================================================

# 모형 1: 무보정 모형
formula1 = 'hp ~ C(dm, Treatment(reference=0))'

# 모형 2: DAG 기반 7개 변수 보정 모형
formula2 = (
    'hp ~ C(dm, Treatment(reference=0))'
    ' + age_num + C(sex_cat) + C(incm_cat)'
    ' + C(edu_cat) + C(smk_cat)'
    ' + C(drk_cat) + C(obe_cat)'
)

# 모형 1: 무보정 로지스틱 회귀분석
model1 = smf.glm(
    formula=formula1,
    data=reg_df,
    family=sm.families.Binomial(),
    freq_weights=reg_df['wt_itvex']
).fit(
    cov_type='cluster',
    cov_kwds={'groups': reg_df['psu']}
)

# 모형 2: 보정 로지스틱 회귀분석
model2 = smf.glm(
    formula=formula2,
    data=reg_df,
    family=sm.families.Binomial(),
    freq_weights=reg_df['wt_itvex']
).fit(
    cov_type='cluster',
    cov_kwds={'groups': reg_df['psu']}
)


# ==============================================================================
# 5. 오즈비, 95% 신뢰구간 및 p값 추출
# ==============================================================================

def get_dm_result(model, model_name):
    # dm 변수가 float형태(0.0, 1.0)이므로 카테고리 명칭이 [T.1.0]으로 생성됩니다.
    term = 'C(dm, Treatment(reference=0))[T.1.0]'

    coef = model.params[term]
    se = model.bse[term]

    or_value = np.exp(coef)
    ci_lower = np.exp(coef - 1.96 * se)
    ci_upper = np.exp(coef + 1.96 * se)
    p_value = model.pvalues[term]

    return {
        '모형': model_name,
        '변수': '당뇨병 유병 (기준: 비유병)',
        'OR': round(or_value, 3),
        '95% CI': f'{ci_lower:.3f} - {ci_upper:.3f}',
        'p값': f'{p_value:.4f}',
        '분석 인원': len(reg_df)
    }


# 당뇨병 변수의 결과만 추출
table2 = pd.DataFrame([
    get_dm_result(model1, '모형 1 (무보정)'),
    get_dm_result(model2, '모형 2 (보정)')
])

print('\n========== Table 2 회귀분석 결과 ==========')
display(table2)


# ==============================================================================
# 6. 회귀분석 결과 및 분석 인원 출력
# ==============================================================================

print('\n[모형 1 요약]')
print(model1.summary())

print('\n[모형 2 요약]')
print(model2.summary())

print('\n========== 최종 분석 인원 ==========')
print('Table 1 분석 대상자:', len(sub_df))
print('모형 1 분석 인원:', int(model1.nobs))
print('모형 2 분석 인원:', int(model2.nobs))
print('결측 등으로 제외된 인원:', len(sub_df) - len(reg_df))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
원자료 불러오기 완료
파일 위치: /content/drive/MyDrive/hn24_all.sas7bdat
원자료 크기: (6997, 12)
--- 분석 대상자 확인 ---
Table 1 분석 대상자: 5567
회귀분석 최종 대상자: 5367
보정변수 결측 등으로 제외된 인원: 200

당뇨병 유병 여부
dm
0.0    4477
1.0     890
Name: count, dtype: int64

고혈압 유병 여부
hp
0.0    3492
1.0    1875
Name: count, dtype: int64


/usr/local/lib/python3.13/dist-packages/statsmodels/genmod/generalized_linear_model.py:1488: SpecificationWarning: cov_type not fully supported with freq_weights
  glm_results = GLMResults(
/usr/local/lib/python3.13/dist-packages/statsmodels/genmod/generalized_linear_model.py:1488: SpecificationWarning: cov_type not fully supported with freq_weights
  glm_results = GLMResults(



========== Table 2 회귀분석 결과 ==========


,모형,변수,OR,95% CI,p값,분석 인원
0,모형 1 (무보정),당뇨병 유병 (기준: 비유병),5.372,4.454 - 6.480,0.0000,5367
1,모형 2 (보정),당뇨병 유병 (기준: 비유병),2.356,1.915 - 2.898,0.0000,5367



[모형 1 요약]
                 Generalized Linear Model Regression Results                  
Dep. Variable:                     hp   No. Observations:                 5367
Model:                            GLM   Df Residuals:              39707663.08
Model Family:                Binomial   Df Model:                            1
Link Function:                  Logit   Scale:                          1.0000
Method:                          IRLS   Log-Likelihood:            -2.2671e+07
Date:                Wed, 11 Nov 2026   Deviance:                   4.5342e+07
Time:                        05:49:36   Pearson chi2:                 3.97e+07
No. Iterations:                     5   Pseudo R-squ. (CS):              1.000
Covariance Type:              cluster                                         
                                           coef    std err          z      P>|z|      [0.025      0.975]
----------------------------------------------------------------------------------------------